In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [28]:
ord = spark.read.csv('/content/drive/MyDrive/Spark_Practice/RetailDB_SalesData/Orders/part-00000', schema = ('order_id int, order_date timestamp, order_customer_id int, order_status string'))
ord.printSchema()
ord.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-------------------+-----------------+---------------+
|order_id|         order_date|order_customer_id|   order_status|
+--------+-------------------+-----------------+---------------+
|       1|2013-07-25 00:00:00|            11599|         CLOSED|
|       2|2013-07-25 00:00:00|              256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|            12111|       COMPLETE|
|       4|2013-07-25 00:00:00|             8827|         CLOSED|
|       5|2013-07-25 00:00:00|            11318|       COMPLETE|
+--------+-------------------+-----------------+---------------+
only showing top 5 rows


### Sort function

In [30]:
# Sort single column
ord.sort(ord.order_customer_id.asc()).show(10)

+--------+-------------------+-----------------+---------------+
|order_id|         order_date|order_customer_id|   order_status|
+--------+-------------------+-----------------+---------------+
|   22945|2013-12-13 00:00:00|                1|       COMPLETE|
|   33865|2014-02-18 00:00:00|                2|       COMPLETE|
|   15192|2013-10-29 00:00:00|                2|PENDING_PAYMENT|
|   57963|2013-08-02 00:00:00|                2|        ON_HOLD|
|   67863|2013-11-30 00:00:00|                2|       COMPLETE|
|   35158|2014-02-26 00:00:00|                3|       COMPLETE|
|   57617|2014-07-24 00:00:00|                3|       COMPLETE|
|   22646|2013-12-11 00:00:00|                3|       COMPLETE|
|   23662|2013-12-19 00:00:00|                3|       COMPLETE|
|   56178|2014-07-15 00:00:00|                3|        PENDING|
+--------+-------------------+-----------------+---------------+
only showing top 10 rows


In [33]:
# Sort multiple columns
ord.sort(ord.order_customer_id.asc(), ord.order_id.desc()).show(10)

+--------+-------------------+-----------------+---------------+
|order_id|         order_date|order_customer_id|   order_status|
+--------+-------------------+-----------------+---------------+
|   22945|2013-12-13 00:00:00|                1|       COMPLETE|
|   67863|2013-11-30 00:00:00|                2|       COMPLETE|
|   57963|2013-08-02 00:00:00|                2|        ON_HOLD|
|   33865|2014-02-18 00:00:00|                2|       COMPLETE|
|   15192|2013-10-29 00:00:00|                2|PENDING_PAYMENT|
|   61453|2013-12-14 00:00:00|                3|       COMPLETE|
|   57617|2014-07-24 00:00:00|                3|       COMPLETE|
|   56178|2014-07-15 00:00:00|                3|        PENDING|
|   46399|2014-05-09 00:00:00|                3|     PROCESSING|
|   35158|2014-02-26 00:00:00|                3|       COMPLETE|
+--------+-------------------+-----------------+---------------+
only showing top 10 rows


In [35]:
# Sort multiple columns - Using ascending parameter
# ascending=[True, False] -> 1st column is asc and 2nd column is desc
# Note: The length of ascending parameter list should be equal to no. of columns involved in sorting
ord.sort(ord.order_customer_id, ord.order_id, ascending=[True, False]).show(10)

+--------+-------------------+-----------------+---------------+
|order_id|         order_date|order_customer_id|   order_status|
+--------+-------------------+-----------------+---------------+
|   22945|2013-12-13 00:00:00|                1|       COMPLETE|
|   67863|2013-11-30 00:00:00|                2|       COMPLETE|
|   57963|2013-08-02 00:00:00|                2|        ON_HOLD|
|   33865|2014-02-18 00:00:00|                2|       COMPLETE|
|   15192|2013-10-29 00:00:00|                2|PENDING_PAYMENT|
|   61453|2013-12-14 00:00:00|                3|       COMPLETE|
|   57617|2014-07-24 00:00:00|                3|       COMPLETE|
|   56178|2014-07-15 00:00:00|                3|        PENDING|
|   46399|2014-05-09 00:00:00|                3|     PROCESSING|
|   35158|2014-02-26 00:00:00|                3|       COMPLETE|
+--------+-------------------+-----------------+---------------+
only showing top 10 rows


### sortWithinPartitions() function

In [50]:
# Sample data
data = [('a', 3), ('b', 1), ('c', 5), ('d', 2), ('e', 4)]
df = spark.createDataFrame(data, ('col1 string, col2 int'))
df.printSchema()
df.show()
df.rdd.glom().collect() # To check data distribution in each partition

root
 |-- col1: string (nullable = true)
 |-- col2: integer (nullable = true)

+----+----+
|col1|col2|
+----+----+
|   a|   3|
|   b|   1|
|   c|   5|
|   d|   2|
|   e|   4|
+----+----+



[[Row(col1='a', col2=3), Row(col1='b', col2=1)],
 [Row(col1='c', col2=5), Row(col1='d', col2=2), Row(col1='e', col2=4)]]

In [47]:
# Using sort - Sorts globally
df.sort(df.col2.asc()).show()

+----+----+
|col1|col2|
+----+----+
|   b|   1|
|   d|   2|
|   a|   3|
|   e|   4|
|   c|   5|
+----+----+



In [48]:
# Using sortWithinPartitions - Sorts data in each partition
df.sortWithinPartitions(df.col2.asc()).show()

+----+----+
|col1|col2|
+----+----+
|   b|   1|
|   a|   3|
|   d|   2|
|   e|   4|
|   c|   5|
+----+----+

